In [1]:
from google.colab import files

uploaded = files.upload()

Saving time_series_60min_singleindex.csv to time_series_60min_singleindex.csv


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('time_series_60min_singleindex.csv')

energy = pd.DataFrame()

energy['date'] = pd.to_datetime(df['utc_timestamp'])

energy['price'] = df['DE_LU_price_day_ahead']
energy['load'] = df['DE_load_actual_entsoe_transparency']
energy['wind'] = df['DE_wind_generation_actual']
energy['solar'] = df['DE_solar_generation_actual']

energy = energy.dropna()

energy = energy.set_index('date')

print(energy.shape)
energy.head()

(17540, 4)


,price,load,wind,solar
date,,,,
2018-09-30 23:00:00+00:00,56.10,42126.0,6042.0,0.0
2018-10-01 00:00:00+00:00,51.41,41500.0,6021.0,0.0
2018-10-01 01:00:00+00:00,47.38,42353.0,6342.0,0.0
2018-10-01 02:00:00+00:00,47.59,43802.0,7144.0,0.0
2018-10-01 03:00:00+00:00,51.61,48065.0,7855.0,0.0


In [3]:
#spike label
threshold = energy['price'].quantile(0.95)

print(threshold)

energy['spike'] = (
    energy['price'] > threshold
).astype(int)

print(
    energy['spike']
    .value_counts()
)

63.900499999999994
spike
0    16663
1      877
Name: count, dtype: int64


Top 5% prices
        =
Price Spike

Άρα μετατρέπουμε το πρόβλημα σε:

0 = normal market
1 = extreme market

Θέλουμε να δημιουργήσουμε χαρακτηριστικά (features) που θα βοηθήσουν το μοντέλο να αναγνωρίσει πότε πλησιάζει ένα spike.

Στην πραγματικότητα προσπαθούμε να απαντήσουμε:

"Τι συμβαίνει στην αγορά λίγο πριν εμφανιστεί ένα ακραίο γεγονός;"

In [4]:
#Feature Engineering
energy['price_lag_1'] = energy['price'].shift(1)
energy['price_lag_2'] = energy['price'].shift(2)
energy['price_lag_3'] = energy['price'].shift(3)
energy['price_lag_24'] = energy['price'].shift(24)

energy['load_lag_1'] = energy['load'].shift(1)
energy['wind_lag_1'] = energy['wind'].shift(1)
energy['solar_lag_1'] = energy['solar'].shift(1)

energy['rolling_mean_24'] = energy['price'].rolling(24).mean()
energy['rolling_std_24'] = energy['price'].rolling(24).std()

energy['hour'] = energy.index.hour
energy['dayofweek'] = energy.index.dayofweek
energy['month'] = energy.index.month

energy = energy.dropna()

print(energy.shape)
energy.head()

(17516, 17)


,price,load,wind,solar,spike,price_lag_1,price_lag_2,price_lag_3,price_lag_24,load_lag_1,wind_lag_1,solar_lag_1,rolling_mean_24,rolling_std_24,hour,dayofweek,month
date,,,,,,,,,,,,,,,,,
2018-10-01 23:00:00+00:00,44.10,46297.0,15895.0,0.0,0,45.10,42.50,46.46,56.10,47735.0,16601.0,0.0,60.143333,12.591821,23,0,10
2018-10-02 00:00:00+00:00,44.06,44899.0,15597.0,0.0,0,44.10,45.10,42.50,51.41,46297.0,15895.0,0.0,59.837083,12.899095,0,1,10
2018-10-02 01:00:00+00:00,43.77,45550.0,16534.0,0.0,0,44.06,44.10,45.10,47.38,44899.0,15597.0,0.0,59.686667,13.070581,1,1,10
2018-10-02 02:00:00+00:00,44.29,47150.0,17883.0,0.0,0,43.77,44.06,44.10,47.59,45550.0,16534.0,0.0,59.549167,13.219873,2,1,10
2018-10-02 03:00:00+00:00,48.13,51794.0,19347.0,0.0,0,44.29,43.77,44.06,51.61,47150.0,17883.0,0.0,59.404167,13.329370,3,1,10


In [5]:
#Train/Test Split
train_size = int(len(energy)*0.75)

train = energy.iloc[:train_size]
test = energy.iloc[train_size:]

print(train.shape)
print(test.shape)

(13137, 17)
(4379, 17)


In [6]:
#x and y variables
features = [
    'load',
    'wind',
    'solar',
    'price_lag_1',
    'price_lag_2',
    'price_lag_3',
    'price_lag_24',
    'load_lag_1',
    'wind_lag_1',
    'solar_lag_1',
    'rolling_mean_24',
    'rolling_std_24',
    'hour',
    'dayofweek',
    'month'
]

X_train = train[features]
X_test = test[features]

y_train = train['spike']
y_test = test['spike']

print(X_train.shape)
print(X_test.shape)
print(y_train.value_counts())
print(y_test.value_counts())

(13137, 15)
(4379, 15)
spike
0    12346
1      791
Name: count, dtype: int64
spike
0    4302
1      77
Name: count, dtype: int64


In [8]:
#Logistic Regression
#Standardization
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [9]:
#Train Logistic Regression
from sklearn.linear_model import LogisticRegression

logit = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

logit.fit(X_train_scaled, y_train)

pred = logit.predict(X_test_scaled)
prob = logit.predict_proba(X_test_scaled)[:,1]

In [10]:
#Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print("F1:", f1_score(y_test,pred))
print("ROC AUC:", roc_auc_score(y_test,prob))

print()
print(confusion_matrix(y_test,pred))

Accuracy: 0.9876684174469057
Precision: 0.5905511811023622
Recall: 0.974025974025974
F1: 0.7352941176470589
ROC AUC: 0.9979834205775628

[[4250   52]
 [   2   75]]


Recall = 97.4%

Αυτό σημαίνει:

Από τα 77 πραγματικά spikes,
το μοντέλο βρήκε τα 75.

Δηλαδή έχασε μόνο:

2 spikes

Για risk management αυτό είναι εξαιρετικό.

Precision = 59%

Αυτό σημαίνει:

Όταν το μοντέλο λέει "έρχεται spike",
έχει δίκιο περίπου στο 60% των περιπτώσεων.

Για rare event prediction αυτό είναι πολύ καλό.

Confusion Matrix
|            | Pred No | Pred Yes |
| ---------- | ------- | -------- |
| Actual No  | 4250    | 52       |
| Actual Yes | 2       | 75       |
Στο energy trading συνήθως προτιμάμε:

λίγα False Negatives
παρά λίγα False Positives

γιατί είναι χειρότερο να χάσεις ένα πραγματικό spike.
ROC-AUC = 0.998

Αυτό είναι σχεδόν τέλειο.

Πρακτικά σημαίνει:

το dataset είναι εξαιρετικά προβλέψιμο όσον αφορά τα extreme price events.

In [11]:
#Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

pred = rf.predict(X_test)
prob = rf.predict_proba(X_test)[:,1]

In [12]:
#evaluation of random forest
print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print("F1:", f1_score(y_test,pred))
print("ROC AUC:", roc_auc_score(y_test,prob))

print(confusion_matrix(y_test,pred))

Accuracy: 0.9910938570449874
Precision: 0.7065217391304348
Recall: 0.8441558441558441
F1: 0.7692307692307693
ROC AUC: 0.9958823138739457
[[4275   27]
 [  12   65]]


In [13]:
#Feature Importance
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': rf.feature_importances_
})

print(
    importance
    .sort_values(
        'importance',
        ascending=False
    )
    .head(15)
)

            feature  importance
3       price_lag_1    0.324374
4       price_lag_2    0.166652
10  rolling_mean_24    0.152068
6      price_lag_24    0.086248
5       price_lag_3    0.068079
0              load    0.050730
11   rolling_std_24    0.028135
7        load_lag_1    0.027661
1              wind    0.021706
12             hour    0.021363
8        wind_lag_1    0.020268
14            month    0.012569
2             solar    0.009994
9       solar_lag_1    0.006450
13        dayofweek    0.003702


Logistic Regression

Προτιμά να βρίσκει σχεδόν όλα τα spikes:

75/77 spikes

αλλά δημιουργεί αρκετά false alarms:

52 false positives
Random Forest

Είναι πιο συντηρητικό:

65/77 spikes

αλλά μειώνει δραματικά τα false alarms:

27 false positives

Για έναν Energy Trader ποιο είναι καλύτερο;

Εξαρτάται από το use case:

Risk Management

Προτιμάμε:

High Recall

άρα:

Logistic Regression

Trading Signal Generation

Προτιμάμε:

High Precision

άρα:

Random Forest

Electricity price spikes in Germany exhibit strong temporal persistence and regime dependence, while fundamental variables such as demand and renewable generation play a secondary but statistically significant role.

In [14]:
#XGBoost Classifier
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    objective='binary:logistic',
    eval_metric='auc',
    random_state=42
)

xgb.fit(X_train, y_train)

pred = xgb.predict(X_test)
prob = xgb.predict_proba(X_test)[:,1]

In [15]:
print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print("F1:", f1_score(y_test,pred))
print("ROC AUC:", roc_auc_score(y_test,prob))

print(confusion_matrix(y_test,pred))

Accuracy: 0.9938342087234528
Precision: 0.8289473684210527
Recall: 0.8181818181818182
F1: 0.8235294117647058
ROC AUC: 0.9976604056101965
[[4289   13]
 [  14   63]]


In [16]:
importance = pd.DataFrame({
    'feature': X_train.columns,
    'importance': xgb.feature_importances_
})

print(
    importance
    .sort_values(
        'importance',
        ascending=False
    )
    .head(15)
)

            feature  importance
3       price_lag_1    0.634204
12             hour    0.060107
4       price_lag_2    0.056157
2             solar    0.033873
10  rolling_mean_24    0.025762
11   rolling_std_24    0.022809
1              wind    0.022571
0              load    0.022486
9       solar_lag_1    0.021519
7        load_lag_1    0.019641
5       price_lag_3    0.017747
13        dayofweek    0.017230
6      price_lag_24    0.016155
8        wind_lag_1    0.016130
14            month    0.013610


Τι επιδιώκουμε με αυτόν τον κώδικα;

Να ελέγξουμε αν ένα πιο εξελιγμένο μοντέλο machine learning μπορεί:

να διατηρήσει το πολύ υψηλό Recall,
να αυξήσει ακόμα περισσότερο το Precision,
και να μας δώσει μια πιο ακριβή εικόνα των παραγόντων που οδηγούν σε electricity price spikes

Σύγκριση των τριών μοντέλων
| Model               | Accuracy  | Precision | Recall    | F1        | ROC-AUC   |
| ------------------- | --------- | --------- | --------- | --------- | --------- |
| Logistic Regression | 0.988     | 0.591     | **0.974** | 0.735     | **0.998** |
| Random Forest       | 0.991     | 0.707     | 0.844     | 0.769     | 0.996     |
| XGBoost             | **0.994** | **0.829** | 0.818     | **0.824** | 0.998     |


Logistic Regression
Εντόπισε σχεδόν όλα τα spikes.
Έδωσε όμως αρκετά false alarms.
75/77 spikes
52 false positives
Random Forest
Μείωσε σημαντικά τα false positives.
65/77 spikes
27 false positives
XGBoost

Το XGBoost πέτυχε το καλύτερο tradeoff:

63/77 spikes
13 false positives

δηλαδή:

όταν το μοντέλο προβλέπει spike, έχει δίκιο στο 83% των περιπτώσεων.

We developed machine learning classifiers to predict extreme electricity price events in the German day-ahead market. The XGBoost model achieved 99.4% accuracy, 82.9% precision, 81.8% recall, and a ROC-AUC of 0.998. Results indicate that electricity price spikes exhibit strong temporal persistence, intraday seasonality, and dependence on renewable generation patterns.